### Tokenization

In [3]:
text="Roman has a many connections on linkedn."
from nltk.tokenize import word_tokenize
words=word_tokenize(text)
print("text:",text)
print("words:",words)

text: Roman has a many connections on linkedn.
words: ['Roman', 'has', 'a', 'many', 'connections', 'on', 'linkedn', '.']


### classify tweet 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Vectorize tweets
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['tweet'])
y = df['sentiment']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Example tweet to predict
new_tweets = [
   "I absolutely love this app!",
    "Not bad, could be better.",
    "Completely useless and frustrating experience."
]

# Vectorize new tweet(s)
X_new = vectorizer.transform(new_tweets)

# Predict
predictions = model.predict(X_new)

# Show results
for tweet, prediction in zip(new_tweets, predictions):
    print(f"Tweet: \"{tweet}\" → Predicted Sentiment: {prediction}")


In [33]:
# Simple Sentiment Analysis (Positive / Neutral / Negative)
import pandas as pd, re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt

# Sample data
data = {
    'tweet': [
        "I love this phone!", "Worst service ever!", "It’s okay, nothing special.",
        "Fantastic experience!", "Delivery was late but okay.",
        "Terrible support.", "So happy with it!", "Just average quality.",
        "Horrible product!", "It’s fine, I don’t care."
    ],
    'sentiment': [
        'Positive','Negative','Neutral','Positive','Neutral',
        'Negative','Positive','Neutral','Negative','Neutral'
    ]
}
df = pd.DataFrame(data)

# Clean text
df['tweet'] = df['tweet'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x.lower()))

# Split data
X_train, X_test, y_train, y_test = train_test_split(df['tweet'], df['sentiment'], test_size=0.3, random_state=42)

# TF-IDF + Model
vec = TfidfVectorizer(stop_words='english')
X_train_tfidf, X_test_tfidf = vec.fit_transform(X_train), vec.transform(X_test)
model = LogisticRegression(max_iter=1000).fit(X_train_tfidf, y_train)

# Predict & Evaluate
y_pred = model.predict(X_test_tfidf)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("\n🧩 Confusion Matrix:\n", cm)

# Test on new tweets
new_tweets = [
    "I hate how slow this app is!",
    "This update is amazing!",
    "It's fine, nothing special."
]
new_clean = [re.sub(r'[^a-zA-Z\s]', '', t.lower()) for t in new_tweets]
preds = model.predict(vec.transform(new_clean))

for t, p in zip(new_tweets, preds):
    print(f"Tweet: {t} → Sentiment: {p}")



Classification Report:
               precision    recall  f1-score   support

    Negative       0.00      0.00      0.00       3.0
     Neutral       0.00      0.00      0.00       0.0

    accuracy                           0.00       3.0
   macro avg       0.00      0.00      0.00       3.0
weighted avg       0.00      0.00      0.00       3.0


🧩 Confusion Matrix:
 [[0 3]
 [0 0]]
Tweet: I hate how slow this app is! → Sentiment: Neutral
Tweet: This update is amazing! → Sentiment: Neutral
Tweet: It's fine, nothing special. → Sentiment: Neutral


C:\Users\SUSHANT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\SUSHANT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\SUSHANT\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\SUSHANT\anaconda3\Lib

### remove stop word

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
filtered_words = [word for word in words if word.lower() not in stop_words and word.isalpha()]
print("no stopwords:", filtered_words)


### svm

In [29]:
# ✅ Sentiment Analysis on IMDB dataset using SVM (Fixed Version)

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.datasets import imdb

# Step 1: Load IMDB data
num_words = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=num_words)

# Step 2: Map integers back to words
word_index = imdb.get_word_index()
index_word = {v + 3: k for k, v in word_index.items()}
index_word[0] = "<PAD>"
index_word[1] = "<START>"
index_word[2] = "<UNK>"
index_word[3] = "<UNUSED>"

def decode_review(encoded_review):
    return ' '.join([index_word.get(i, '?') for i in encoded_review])

# Step 3: Decode a subset for faster training
X_train_text = [decode_review(r) for r in X_train[:5000]]
X_test_text = [decode_review(r) for r in X_test[:2000]]
y_train = y_train[:5000]
y_test = y_test[:2000]

# Step 4: TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

# Step 5: Train SVM
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

# Step 6: Predict
y_pred = svm_model.predict(X_test_tfidf)

# Step 7: Evaluation
print("\n✅ Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

cm = confusion_matrix(y_test, y_pred)
print("\n🧩 Confusion Matrix:\n", cm)

pred_counts = pd.Series(y_pred).value_counts()
print("\n📊 Predicted Sentiment Counts:")
print(f"Positive reviews: {pred_counts.get(1,0)}")
print(f"Negative reviews: {pred_counts.get(0,0)}")


1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 3s 2us/step

✅ Classification Report:

              precision    recall  f1-score   support

    Negative       0.85      0.84      0.85      1047
    Positive       0.83      0.84      0.83       953

    accuracy                           0.84      2000
   macro avg       0.84      0.84      0.84      2000
weighted avg       0.84      0.84      0.84      2000


🧩 Confusion Matrix:
 [[881 166]
 [151 802]]

📊 Predicted Sentiment Counts:
Positive reviews: 968
Negative reviews: 1032


### different stemming algorithm

In [ ]:
from nltk.stem import PorterStemmer,LancasterStemmer,SnowballStemmer
porter=PorterStemmer()
lancaster=LancasterStemmer()
snowball=SnowballStemmer('english')

p=[porter.stem(word) for word in words]
l=[lancaster.stem(word) for word in words]
s=[snowball.stem(word) for word in words]

print("words:",words)
print('porterstem:',p)
print('lancasterstem:',l)
print('snowball:',s)

### lemmatize the word 

In [ ]:
import spacy
from nltk.stem import PorterStemmer
words=['Roman', 'has', 'a', 'many', 'connections', 'on', 'linkedn', '.']

nlp = spacy.load("en_core_web_sm")
doc = nlp(" ".join(words))

lemmatized = [token.lemma_ for token in doc]
print("Lemmatized Words:", lemmatized)

### pos tag

In [ ]:
import nltk
pos_tag=nltk.pos_tag(words)
noun=[word for word,tag in pos_tag if tag.startswith('NN')]
verb=[word for word,tag in pos_tag if tag.startswith('VB')]
print('tag:',pos_tag)
print("noun:",noun)
print('verb:',verb)

### Bag of word

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

text=['Data science is amazing','machine learning is the part of data science','data science uses python']

vectorizer=CountVectorizer()
bow=vectorizer.fit_transform(text)

print('feature names:',vectorizer.get_feature_names_out(text))
print('bow matrix:',bow.toarray())

### NER

In [ ]:
import spacy
text="Sushant R. Thite pursuing master from MAEER'S MIT Group of institute in Pune."
nlp=spacy.load('en_core_web_sm')
doc=nlp(text)
print("ner:")
for i in doc.ents:
    print(i.text,"-->",i.label_)

### TFIDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
text=['Data science is amazing','machine learning is the part of data science','data science uses python']
vectorizer=TfidfVectorizer()
tfidf=vectorizer.fit_transform(text)

print("feature:",vectorizer.get_feature_names_out(text))
print("tfidf:",tfidf.toarray())

### CYK 

In [ ]:
grammer={
    'S':[('A','B')],
    'A':[('A','A'),('a',)],
    'B':[('B','B'),('b',)]
}
sentence='aaabbb'

def cyk_parse(G,s):
    n=len(s)
    T=[[set() for _ in range(n)]for _ in range(n)]
    for j,ch in enumerate(s):
        T[j][j]={A for A,prods in G.items() for p in prods if p==(ch,)}
    for l in range(2,n+1):
        for i in range (n-l+1):
            j=i+l-1
            for k in range(i,j):
                for A,prods in G.items():
                    T[i][j].update(A for B , C in [p for p in prods if len(p)==2] if B in T[i][k] and C in T[k+1][j])
    for row in T:
        print([list(cell) for cell in row])
    return 'S' in T[0][n-1]

result=cyk_parse(grammer,sentence)
print("reult:","accept" if result else "reject")
                    
                    
            
                    

### early parsing

In [ ]:
from nltk import CFG

grammar = CFG.fromstring("""
S -> NP VP
NP -> Det N | N
VP -> V NP
Det -> 'the' | 'a'
N -> 'dog' | 'cat'
V -> 'chased' | 'saw'
""")

from nltk.parse import EarleyChartParser

parser = EarleyChartParser(grammar)
sentence = "the dog chased the cat".split()

for tree in parser.parse(sentence):
    print(tree)
    tree.pretty_print()


### unigram and bigram

In [ ]:
#a. Implement unigram and bigram models.
import nltk
from collections import defaultdict, Counter
nltk.download('punkt_tab')

# Example corpus (you can expand this)
corpus = [
    "I love machine learning",
    "Machine learning is fun",
    "I enjoy learning new things"
]

# Tokenize
tokens = [nltk.word_tokenize(sent.lower()) for sent in corpus]
flat_tokens = [word for sentence in tokens for word in sentence]

# Count unigrams
unigram_counts = Counter(flat_tokens)
total_unigrams = sum(unigram_counts.values())

# Probability of a word
def unigram_prob(word):
    return unigram_counts[word] / total_unigrams if word in unigram_counts else 0.0

# Example
print("Unigram Probability of 'learning':", unigram_prob('learning'))


#.bigram
# Count bigrams
bigrams = []
for sentence in tokens:
    for i in range(len(sentence) - 1):
        bigrams.append((sentence[i], sentence[i+1]))

bigram_counts = Counter(bigrams)

# Conditional bigram probability
def bigram_prob(w1, w2):
    if unigram_counts[w1] == 0:
        return 0.0
    return bigram_counts[(w1, w2)] / unigram_counts[w1]

# Example
print("Bigram Probability of ('machine', 'learning'):", bigram_prob('machine', 'learning'))


### HMM

In [ ]:
#b.Hidden Markov Model(HMM)

import nltk
from collections import defaultdict
nltk.download('treebank')

# Load tagged sentences
tagged_sentences = nltk.corpus.treebank.tagged_sents()

# Initialize counts
transition_counts = defaultdict(Counter)
emission_counts = defaultdict(Counter)
tag_counts = Counter()

# Count transitions and emissions
for sentence in tagged_sentences:
    previous_tag = "<s>"
    for word, tag in sentence:
        transition_counts[previous_tag][tag] += 1
        emission_counts[tag][word.lower()] += 1
        tag_counts[tag] += 1
        previous_tag = tag

# Transition probability
def transition_prob(t1, t2):
    return transition_counts[t1][t2] / sum(transition_counts[t1].values())

# Emission probability
def emission_prob(tag, word):
    return emission_counts[tag][word.lower()] / tag_counts[tag]

# Example
print("P(NN|DT):", transition_prob('DT', 'NN'))
print("P(dog|NN):", emission_prob('NN', 'dog'))


### BLEU

In [ ]:
# Define reference (actual) and candidate (predicted) sentences
reference = "the cat is on the mat".split()
candidate = "the cat sat on the mat".split()

print("Reference:", reference)
print("Candidate:", candidate)

# Calculate unigram precision manually
from collections import Counter

ref_counts = Counter(reference)
cand_counts = Counter(candidate)

# Count how many candidate words appear in reference
matches = sum(min(cand_counts[word], ref_counts[word]) for word in cand_counts)
unigram_precision = matches / len(candidate)

print("Unigram matches:", matches)
print("Unigram precision:", unigram_precision)

# Calculate bigram precision manually
def get_ngrams(words, n):
    return [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]

ref_bigrams = Counter(get_ngrams(reference, 2))
cand_bigrams = Counter(get_ngrams(candidate, 2))

matches_bigram = sum(min(cand_bigrams[b], ref_bigrams[b]) for b in cand_bigrams)
bigram_precision = matches_bigram / len(get_ngrams(candidate, 2))

print("Bigram matches:", matches_bigram)
print("Bigram precision:", bigram_precision)

# Calculate brevity penalty
import math

c = len(candidate)
r = len(reference)

if c > r:
    BP = 1
else:
    BP = math.exp(1 - (r / c))

print("Brevity Penalty (BP):", BP)

# Combine unigram and bigram precision (equal weights)
w1 = w2 = 0.5

if unigram_precision > 0 and bigram_precision > 0:
    bleu = BP * math.exp((w1 * math.log(unigram_precision)) + (w2 * math.log(bigram_precision)))
else:
    bleu = 0

print("Final BLEU Score (up to 2-grams):", bleu)

### EDA

In [ ]:
#Basic EDA

# Word counts per review
df['word_count'] = df['review'].apply(lambda x: len(x.split()))

# Average review length
avg_len = df['word_count'].mean()

# Top words
from collections import Counter
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

all_words = ' '.join(df['review']).lower().split()
filtered_words = [word for word in all_words if word.isalpha() and word not in stop_words]
top_words = Counter(filtered_words).most_common(10)

# Show stats
print("Average review length (words):", avg_len)
print("Top 10 words:", top_words)

### similar word 

In [ ]:
from gensim.models import Word2Vec
corpus = [
    ['i', 'love', 'machine', 'learning'],
    ['machine', 'learning', 'is', 'fun'],
    ['i', 'enjoy', 'learning', 'new', 'things'],
    ['deep', 'learning', 'is', 'a', 'branch', 'of', 'machine', 'learning']
]

model=Word2Vec(sentences=corpus,vector_size=100,window=5,min_count=1,workers=4)
similar_word=model.wv.most_similar('machine',topn=3)
print("similar word")
for word,score in similar_word:
    print(f"{word} ---> {score:.4f}")